# Supervised Finetuning

This tutorial adapts a pretrained TERRA model to a label transfer classification task using parameter-efficient finetuning (selective unfreezing), starting from a user-provided `AnnData`.

**You will:**
1. Load a labeled demo dataset.
2. Download a pretrained TERRA model from the Hugging Face Hub.
3. Harmonize and tokenize your data against that model.
4. Split into train/validation/test sets.
5. Configure and run finetuning.

:::{note}
**TERRA requires an NVIDIA GPU** for the finetuning step.
:::

New to TERRA? Start with the {doc}`zero-shot quickstart <zero_shot_quickstart>` first.

## 1. Setup

In [ ]:
import logging
from pathlib import Path

import scanpy as sc
from datasets import concatenate_datasets
from sklearn.model_selection import train_test_split

from terra import download_pretrained, harmonize_adata, tokenize_adata
from terra.training.finetune import finetune
from terra.utils import filter_dataset_by_cell_ids

logging.basicConfig(level="INFO")  # see TERRA progress

## 2. Demo data

Download the file at the following link: https://drive.google.com/file/d/1MoK7aVs9m9Oy3uzpXvxNvVZBU_S0wR4q/view?usp=share_link

In [ ]:
# Update this path, if required.
adata_path = sc.read_h5ad("./nanostring_cosmx_human_nsclc_subsample_10pct_batch1.h5ad")

## 3. Download a pretrained TERRA model

`download_pretrained` fetches a self-contained model bundle (checkpoint, tokenizer, and the gene-reference files used for harmonization) and returns the local folder path.

In [ ]:
# By default the model is cached in the Hugging Face cache; pass
# local_dir="terra_model" to download it into a folder of your choice.
model_dir = download_pretrained("Lotfollahi-lab/TERRA-96M")

## 4. Harmonize and tokenize your data

In [ ]:
# Load the adata object from disk
adata = sc.read_h5ad(adata_path)
adata.obs_names_make_unique()

In [ ]:
# Gene-reference files default to the ones shipped inside the downloaded model bundle (`ensembl_dictionary.pkl`, `gene_count_dictionary.pkl`)
# This ensures that harmonization and tokenization reproduces the model's training-time gene mapping and occurrence filtering.
gene_mapping_dict_file_path = str(Path(model_dir) / "ensembl_dictionary.pkl")
gene_occurrence_count_file_path = str(Path(model_dir) / "gene_count_dictionary.pkl")

min_cells_per_gene = 0
min_genes_per_cell = 0

In [ ]:
# If your data has more than one sample/section (set `sample_key` to the `obs` column identifying each one).
# Each sample is harmonized and tokenized separately. 
# For a single sample (`sample_key = None`), harmonization and tokenization each run once on the whole `AnnData`.
sample_key = "sample" if "sample" in adata.obs else None

In [ ]:
if sample_key:
    adata = harmonize_adata(
        adata,
        gene_mapping_dict_file_path=gene_mapping_dict_file_path,
        gene_occurrence_count_file_path=gene_occurrence_count_file_path,
        species="human",
        min_cells_per_gene=0,
        min_genes_per_cell=0,
    )

    datasets_by_sample = []
    idx_list, gene_list = [], []
    for sample in adata.obs[sample_key].unique():
        adata_sample = adata[adata.obs[sample_key] == sample]
        adata_sample = harmonize_adata(
            adata_sample,
            gene_mapping_dict_file_path=gene_mapping_dict_file_path,
            gene_occurrence_count_file_path=gene_occurrence_count_file_path,
            species="human",
            min_cells_per_gene=min_cells_per_gene,
            min_genes_per_cell=min_genes_per_cell,
        )
        idx_list.extend(adata_sample.obs.index.tolist())
        gene_list.extend(adata_sample.var.index.tolist())
        datasets_by_sample.append(tokenize_adata(
            adata_sample,
            model_folder_path=model_dir,
            cache_directory_path="./terra_cache",
        ))

    adata = adata[idx_list, list(set(gene_list))]
    dataset = concatenate_datasets(datasets_by_sample)
else:
    adata = harmonize_adata(
        adata,
        gene_mapping_dict_file_path=gene_mapping_dict_file_path,
        gene_occurrence_count_file_path=gene_occurrence_count_file_path,
        species="human",
        min_cells_per_gene=min_cells_per_gene,
        min_genes_per_cell=min_genes_per_cell,
    )
    # Reads the pretrained model's tokenizer type/config from `model_dir` and
    # preserves `obs["cell_id"]` on each tokenized example.
    dataset = tokenize_adata(
        adata,
        model_folder_path=model_dir,
        cache_directory_path="./terra_cache",
    )

## 5. Split into train, validation, and test sets

This demo dataset splits cells from a single sample into train-val-test sets. For group-aware, label-balanced nested cross-validation (e.g. across patients/samples), see `NestedStratifiedGroupKFold` in `terra.utils.nested_stratified_group_split`.

In [ ]:
adata.obs['cell_id']

In [ ]:
# Set the column containing the labels
label_name = "niche"
labels = adata.obs[label_name]

adata.obs[label_name] = adata.obs[label_name].astype("category")
adata.uns[f"{label_name}_num_classes"] = len(adata.obs[label_name].cat.categories)
adata.obs[label_name] = adata.obs[label_name].cat.codes

In [ ]:
# Create a held-out test set...
train_val_ids, test_ids = train_test_split(
    adata.obs["cell_id"],
    test_size=0.2,
    stratify=labels,
    random_state=0,
)
# Split the remainder into train/val (0.25 x 0.8 = 0.2 -> 60/20/20 overall).
train_ids, val_ids = train_test_split(
    train_val_ids,
    test_size=0.25,
    stratify=labels.loc[train_val_ids.index],
    random_state=0,
)

In [ ]:
train_adata = adata[adata.obs["cell_id"].isin(train_ids)].copy()
val_adata = adata[adata.obs["cell_id"].isin(val_ids)].copy()
test_adata = adata[adata.obs["cell_id"].isin(test_ids)].copy()

train_dataset = filter_dataset_by_cell_ids(dataset, train_ids.tolist(), temp_dir="./terra_cache/train")
val_dataset = filter_dataset_by_cell_ids(dataset, val_ids.tolist(), temp_dir="./terra_cache/val")
test_dataset = filter_dataset_by_cell_ids(dataset, test_ids.tolist(), temp_dir="./terra_cache/test")

## 6. Configure finetuning

`finetune` takes a plain nested `args` dict as input. With `use_peft=False`, the `finetune` module selectively unfreezes the target encoder submodules whose names match `peft_target_modules` while the rest of the pretrained encoder stays frozen. Here, we unfreeze the last encoder block's attention and MLP projections.

:::{note}
LoRA is also available as a parameter-efficient alternative: set `use_peft: True` and provide `peft_rank`, `peft_alpha`, `peft_dropout`, `peft_bias`, and `peft_task_type`.
:::

In [ ]:
args = {
    "model": {
        "pretrained_checkpoint_path": model_dir,
    },
    "data": {
        "label_name": label_name,
        "batch_size": 100,
        "pin_memory": False,
        "num_workers": 4,
    },
    "finetune": {
        # Selective unfreezing (no PEFT): only params whose name contains one
        # of these substrings are trained. This targets the last encoder
        # block (index 11 for a 12-block encoder) -- adjust to match your
        # model's depth.
        "use_peft": False,
        "peft_target_modules": [
            "11.attn.qkv",
            "11.attn.proj",
            "11.mlp.fc1",
            "11.mlp.fc2",
        ],
        "use_mlp": False,
        "hidden_dim": 256,
        "lr": 1e-3,
        "num_epochs": 5,
        "patience": 10,          # early-stop if val loss doesn't improve for this many epochs
        "save_every_k_epochs": -1,
        "selection_type": "agg_graph",
        "excluded_tokens": None,
        "top_k": None,
    },
}

## 7. Run finetuning

Passing `val` and `test` together gives you the standard train/val/test behavior: `val` drives early stopping, while `test` is the final, held-out evaluation set — `finetune` returns per-cell predictions, targets, and accuracy on `test`.

In [ ]:
predictions, targets, accuracy = finetune(
    args=args,
    train_adata=train_adata,
    train_dataset=train_dataset,
    val_adata=val_adata,
    val_dataset=val_dataset,
    test_adata=test_adata,
    test_dataset=test_dataset,
)
print(f"Test accuracy: {accuracy:.3f}")